In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score


In [2]:
dataset= pd.read_csv('Digital_Marketing_DP.csv')
dataset.head()

,Age,Income,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,...,LoyaltyPoints,Conversion,Gender_Male,CampaignChannel_PPC,CampaignChannel_Referral,CampaignChannel_SEO,CampaignChannel_Social Media,CampaignType_Consideration,CampaignType_Conversion,CampaignType_Retention
0,56,136912,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,...,688,1,0,0,0,0,1,0,0,0
1,69,41760,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,...,3459,1,1,0,0,0,0,0,0,1
2,46,88456,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,...,2337,1,0,1,0,0,0,0,0,0
3,32,44085,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,...,2463,1,0,1,0,0,0,0,1,0
4,60,83964,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,...,4345,1,0,1,0,0,0,0,1,0


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb

In [4]:
#  Feature Selection
from sklearn.feature_selection import (
    VarianceThreshold,
    chi2,
    f_classif,
    mutual_info_classif,
    RFE,
    SelectFromModel,
)

In [5]:
cat_cols = dataset.select_dtypes(include=["object"]).columns.tolist()
print(f"\nEncoding categorical columns: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    dataset[col] = le.fit_transform(dataset[col].astype(str))
X = dataset.drop("Conversion", axis=1)
y = dataset["Conversion"]

# Scale features (needed for SVM, LR, KNN; harmless for tree-based)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Also keep min-max scaled (non-negative) for Chi-Square
from sklearn.preprocessing import MinMaxScaler
X_minmax = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape} | Test size: {X_test.shape}")

#Smote for data imbalance
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_smote.value_counts())


Encoding categorical columns: []

Train size: (6400, 21) | Test size: (1600, 21)
Before SMOTE:
 Conversion
1    5610
0     790
Name: count, dtype: int64
After SMOTE:
 Conversion
1    5610
0    5610
Name: count, dtype: int64


In [7]:
feature_scores = pd.DataFrame(index=X.columns)

# 1. Variance Threshold 
print("\n Variance Threshold (threshold = 0.01)")
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_scaled)
vt_mask = vt.get_support()
vt_selected = X.columns[vt_mask].tolist()
print(f"     Features retained: {len(vt_selected)} / {X.shape[1]}")
print(f"     Removed           : {list(set(X.columns) - set(vt_selected))}")
feature_scores["Variance"] = vt.variances_

# 2.  Chi-Square 
print("\n Chi-Square Test")
chi2_scores, chi2_pvals = chi2(X_minmax, y)
chi2_series = pd.Series(chi2_scores, index=X.columns).sort_values(ascending=False)
chi2_selected = chi2_series[chi2_series > chi2_series.median()].index.tolist()
feature_scores["Chi2_Score"] = chi2_scores
feature_scores["Chi2_Rank"]  = chi2_series.rank(ascending=False)
print(f"     Top-5 features:\n{chi2_series.head()}")

#  3.  ANOVA F-test 
print("\n ANOVA F-Test")
f_scores, f_pvals = f_classif(X_scaled, y)
f_series = pd.Series(f_scores, index=X.columns).sort_values(ascending=False)
f_selected = f_series[f_series > f_series.median()].index.tolist()
feature_scores["F_Score"] = f_scores
feature_scores["F_Rank"]  = f_series.rank(ascending=False)
print(f"     Top-5 features:\n{f_series.head()}")

# 4. Mutual Information 
print("\n Mutual Information")
mi_scores = mutual_info_classif(X_scaled, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_selected = mi_series[mi_series > mi_series.median()].index.tolist()
feature_scores["MI_Score"] = mi_scores
feature_scores["MI_Rank"]  = mi_series.rank(ascending=False)
print(f"     Top-5 features:\n{mi_series.head()}")

# 5. Wrapper: RFE with Logistic Regression 
print("\n Recursive Feature Elimination (RFE)")
n_features_rfe = 8
rfe = RFE(
    estimator=LogisticRegression(max_iter=500, random_state=42),
    n_features_to_select=n_features_rfe,
    step=1,
)
rfe.fit(X_scaled, y)
rfe_selected = X.columns[rfe.support_].tolist()
feature_scores["RFE_Selected"] = rfe.support_.astype(int)
feature_scores["RFE_Rank"]     = rfe.ranking_
print(f"     RFE selected ({n_features_rfe} features): {rfe_selected}")

# 6. Embedded: Random Forest Importance
print("\n Random Forest Feature Importance")
rf_sel = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_sel.fit(X_scaled, y)
rf_imp = pd.Series(rf_sel.feature_importances_, index=X.columns).sort_values(ascending=False)
rf_sfm = SelectFromModel(rf_sel, prefit=True, threshold="median")
rf_selected = X.columns[rf_sfm.get_support()].tolist()
feature_scores["RF_Importance"] = rf_sel.feature_importances_
feature_scores["RF_Rank"]       = rf_imp.rank(ascending=False)
print(f"     Top-5 features:\n{rf_imp.head()}")

# 7. Embedded: LASSO (L1 Logistic Regression) 
print("\n LASSO via L1-Regularized Logistic Regression")
lasso = LogisticRegression(penalty="l1", solver="liblinear", C=0.1, max_iter=500, random_state=42)
lasso.fit(X_scaled, y)
lasso_coef = pd.Series(np.abs(lasso.coef_[0]), index=X.columns).sort_values(ascending=False)
lasso_selected = lasso_coef[lasso_coef > 0].index.tolist()
feature_scores["LASSO_Coef"] = np.abs(lasso.coef_[0])
feature_scores["LASSO_Rank"] = lasso_coef.rank(ascending=False)
print(f"     Features with non-zero coefficient ({len(lasso_selected)}): {lasso_selected}")




 Variance Threshold (threshold = 0.01)
     Features retained: 21 / 21
     Removed           : []

 Chi-Square Test
     Top-5 features:
CampaignType_Conversion    62.467044
EmailClicks                27.233651
TimeOnSite                 22.937242
EmailOpens                 22.598078
PreviousPurchases          20.651064
dtype: float64

 ANOVA F-Test
     Top-5 features:
TimeOnSite          136.649486
EmailClicks         136.460212
EmailOpens          126.713047
AdSpend             126.276640
ClickThroughRate    116.878172
dtype: float64

 Mutual Information
     Top-5 features:
TimeOnSite           0.014891
ClickThroughRate     0.014674
PreviousPurchases    0.013157
AdSpend              0.013013
PagesPerVisit        0.012209
dtype: float64

 Recursive Feature Elimination (RFE)
     RFE selected (8 features): ['AdSpend', 'ClickThroughRate', 'PagesPerVisit', 'TimeOnSite', 'EmailOpens', 'EmailClicks', 'PreviousPurchases', 'CampaignType_Conversion']

 Random Forest Feature Importance
   

In [9]:
print("FEATURE RANKING")


rank_cols = ["Chi2_Rank", "F_Rank", "MI_Rank", "RFE_Rank", "RF_Rank", "LASSO_Rank"]
feature_scores["Avg_Rank"] = feature_scores[rank_cols].mean(axis=1)
feature_scores_sorted = feature_scores.sort_values("Avg_Rank")


print("\nConsensus ranking (lower = more important):")
print(feature_scores_sorted[["Avg_Rank"] + rank_cols].round(2).to_string())

#print(feature_scores_sorted[rank_cols].round(2).to_string())

# Final feature set: top-8 by consensus
TOP_N = 8
final_features = feature_scores_sorted.head(TOP_N).index.tolist()
print(f"\n Final selected features (top {TOP_N}): {final_features}")

X_train_sel = X_train[final_features]
X_test_sel  = X_test[final_features]


FEATURE RANKING

Consensus ranking (lower = more important):
                              Avg_Rank  Chi2_Rank  F_Rank  MI_Rank  RFE_Rank  RF_Rank  LASSO_Rank
TimeOnSite                        1.67        3.0     1.0      1.0         1      2.0         2.0
AdSpend                           3.67        6.0     4.0      4.0         1      4.0         3.0
ClickThroughRate                  4.00        7.0     5.0      2.0         1      5.0         4.0
EmailClicks                       4.67        2.0     2.0     11.0         1     11.0         1.0
EmailOpens                        4.83        4.0     3.0      8.0         1      8.0         5.0
PagesPerVisit                     5.00        8.0     7.0      5.0         1      1.0         8.0
PreviousPurchases                 5.17        5.0     6.0      3.0         1      9.0         7.0
CampaignType_Conversion           6.50        1.0     8.0      9.0         1     14.0         6.0
ConversionRate                    7.17       10.0    10.0

In [51]:
X_train_sel.head()

,CustomerID,TimeOnSite,ClickThroughRate,AdSpend,EmailClicks,PreviousPurchases,EmailOpens,PagesPerVisit
2787,-0.525028,1.089026,1.583129,0.902711,-1.563996,0.178156,1.317361,-0.492532
7093,1.339525,1.198075,-1.560964,0.219354,-1.563996,-0.168115,-0.608829,1.322783
6379,1.030354,-0.522276,-1.591065,-1.651582,-0.163625,0.524427,1.142253,0.580186
3865,-0.058240,1.508414,0.766031,0.660752,1.236747,1.216969,-0.608829,0.821299
1167,-1.226508,-1.442641,-1.265255,-1.033073,1.586840,0.524427,-0.258613,1.075372


In [52]:
X_test_sel.head()

,CustomerID,TimeOnSite,ClickThroughRate,AdSpend,EmailClicks,PreviousPurchases,EmailOpens,PagesPerVisit
3344,-0.283840,1.537922,-0.924670,-1.030498,-0.163625,-0.860656,0.266712,0.870493
408,-1.555165,1.019818,-0.694762,-0.606758,0.886654,-0.168115,0.616928,1.386149
5038,0.449684,1.171054,0.557767,-0.980297,1.236747,-0.860656,-0.608829,-0.446829
6499,1.082315,-1.347905,-0.402348,-0.618719,-1.563996,1.563240,-0.258613,-1.540982
4266,0.115398,1.688105,-0.837274,-0.422269,1.236747,-0.168115,0.616928,0.886220


In [44]:
# 6. CLASSIFICATION MODELS


models = {
    "Logistic Regression"     : LogisticRegression(max_iter=500, random_state=42),
    "Decision Tree"           : DecisionTreeClassifier(random_state=42),
    "Random Forest"           : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting"       : GradientBoostingClassifier(n_estimators=200, random_state=42),
    "XGBoost"                 : xgb.XGBClassifier(n_estimators=200, use_label_encoder=False,
                                                   eval_metric="logloss", random_state=42, verbosity=0),
    "AdaBoost"                : AdaBoostClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)"               : SVC(probability=True, random_state=42),
    "K-Nearest Neighbors"     : KNeighborsClassifier(n_neighbors=7),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    cv_roc  = cross_val_score(model, X_train_sel, y_train, cv=cv, scoring="roc_auc",  n_jobs=-1)
    cv_acc  = cross_val_score(model, X_train_sel, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    cv_f1   = cross_val_score(model, X_train_sel, y_train, cv=cv, scoring="f1",       n_jobs=-1)

    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    results[name] = {
        "CV_AUC_Mean"  : cv_roc.mean(),
        "CV_AUC_Std"   : cv_roc.std(),
        "CV_Acc_Mean"  : cv_acc.mean(),
        "CV_F1_Mean"   : cv_f1.mean(),
        "Test_AUC"     : roc_auc_score(y_test, y_prob),
        "Test_Accuracy": accuracy_score(y_test, y_pred),
        "model"        : model,
        "y_pred"       : y_pred,
        "y_prob"       : y_prob,
    }
    print(f"\n{name}")
    print(f"  CV AUC  : {cv_roc.mean():.4f}")
    print(f"  CV Acc  : {cv_acc.mean():.4f}")
    print(f"  CV F1   : {cv_f1.mean():.4f}")
    print(f"  Test AUC: {roc_auc_score(y_test, y_prob):.4f}  |  Test Acc: {accuracy_score(y_test, y_pred):.4f}")



Logistic Regression
  CV AUC  : 0.8133
  CV Acc  : 0.8805
  CV F1   : 0.9352
  Test AUC: 0.8005  |  Test Acc: 0.8756

Decision Tree
  CV AUC  : 0.6842
  CV Acc  : 0.8611
  CV F1   : 0.9206
  Test AUC: 0.6419  |  Test Acc: 0.8438

Random Forest
  CV AUC  : 0.9092
  CV Acc  : 0.8866
  CV F1   : 0.9378
  Test AUC: 0.9076  |  Test Acc: 0.8900

Gradient Boosting
  CV AUC  : 0.9158
  CV Acc  : 0.8905
  CV F1   : 0.9392
  Test AUC: 0.9162  |  Test Acc: 0.8881

XGBoost
  CV AUC  : 0.9005
  CV Acc  : 0.8873
  CV F1   : 0.9369
  Test AUC: 0.9014  |  Test Acc: 0.8881

AdaBoost
  CV AUC  : 0.9107
  CV Acc  : 0.8945
  CV F1   : 0.9419
  Test AUC: 0.9042  |  Test Acc: 0.8850

SVM (RBF)
  CV AUC  : 0.8484
  CV Acc  : 0.8869
  CV F1   : 0.9392
  Test AUC: 0.8286  |  Test Acc: 0.8781

K-Nearest Neighbors
  CV AUC  : 0.7748
  CV Acc  : 0.8817
  CV F1   : 0.9358
  Test AUC: 0.7948  |  Test Acc: 0.8806
